# Lab 12: Breaking the O(n^2) Barrier — Analysis

In this notebook you'll visualize how advanced sorting algorithms work, measure their performance against the simple sorts from Lab 11, and discover quicksort's weakness.

**Before you start:** Paste your completed sort functions into the cell below.

In [ ]:
# ── Paste your three completed functions here ────────────────────
# You need: _gap_insertion_sort, shell_sort (provided),
# merge_sort (with your merge step), _partition,
# _quick_sort_helper (provided), quick_sort (provided),
# merge_sort_counted (provided), quick_sort_counted (provided)
#
# Easiest approach: paste the ENTIRE sorting.py file contents here.


In [ ]:
# Quick sanity check — run this to make sure your functions work
assert shell_sort([5, 3, 1, 4, 2]) == [1, 2, 3, 4, 5], "shell_sort failed"
assert merge_sort([5, 3, 1, 4, 2]) == [1, 2, 3, 4, 5], "merge_sort failed"
assert quick_sort([5, 3, 1, 4, 2]) == [1, 2, 3, 4, 5], "quick_sort failed"

r1, c1, m1 = merge_sort_counted([3, 1, 2])
assert r1 == [1, 2, 3], "merge_sort_counted failed"

r2, c2, e2 = quick_sort_counted([3, 1, 2])
assert r2 == [1, 2, 3], "quick_sort_counted failed"

print("All functions working!")

---
## Experiment 1: Watching the Mechanism

Before we measure performance, let's *see* how each algorithm sorts. The helper functions below print the list state at key moments so you can watch the progress.

In [ ]:
def shell_sort_visual(a_list):
    """Shell sort with printout after each gap pass."""
    a_list = a_list[:]
    n = len(a_list)
    gap = n // 2
    print(f"Start:    {a_list}  (n={n})")
    while gap > 0:
        for start in range(gap):
            for i in range(start + gap, n, gap):
                val = a_list[i]
                pos = i
                while pos >= gap and a_list[pos - gap] > val:
                    a_list[pos] = a_list[pos - gap]
                    pos -= gap
                a_list[pos] = val
        print(f"Gap {gap:>2}:   {a_list}")
        gap //= 2
    return a_list


def merge_sort_visual(a_list, depth=0):
    """Merge sort with printout showing split and merge at each level."""
    indent = "  " * depth
    if len(a_list) <= 1:
        return a_list
    mid = len(a_list) // 2
    print(f"{indent}Split: {a_list} -> {a_list[:mid]} + {a_list[mid:]}")
    left = merge_sort_visual(a_list[:mid], depth + 1)
    right = merge_sort_visual(a_list[mid:], depth + 1)
    # merge
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    print(f"{indent}Merge: {left} + {right} -> {result}")
    return result


def quick_sort_visual(a_list):
    """Quicksort with printout showing each partition result."""
    a_list = a_list[:]
    def _qs(lst, first, last):
        if first < last:
            piv = lst[first]
            lm, rm = first + 1, last
            done = False
            while not done:
                while lm <= rm and lst[lm] <= piv: lm += 1
                while lm <= rm and lst[rm] >= piv: rm -= 1
                if rm < lm: done = True
                else: lst[lm], lst[rm] = lst[rm], lst[lm]
            lst[first], lst[rm] = lst[rm], lst[first]
            print(f"  Pivot {piv:>2} -> final index {rm}: {lst}")
            _qs(lst, first, rm - 1)
            _qs(lst, rm + 1, last)
    print(f"Start: {a_list}")
    _qs(a_list, 0, len(a_list) - 1)
    return a_list

In [ ]:
test_list = [54, 26, 93, 17, 77, 31, 44, 55, 20]

print("=" * 55)
print("SHELL SORT")
print("=" * 55)
shell_sort_visual(test_list)

print()
print("=" * 55)
print("MERGE SORT")
print("=" * 55)
merge_sort_visual(test_list)

print()
print("=" * 55)
print("QUICKSORT")
print("=" * 55)
quick_sort_visual(test_list)

### Experiment 1 Questions

Study the output above and answer these questions:

**Q1:** In shell sort, look at the list after the first gap pass vs. after the last. Is it fully sorted before gap = 1, or just "closer to sorted"? What does that tell you about what the gap = 1 pass has to do?

*Your answer:*


**Q2:** In merge sort, how many levels of splitting happen for 9 items? Does the merge step at each level touch every item in that subproblem?

*Your answer:*


**Q3:** In quicksort, how many items are permanently placed after each partition call? How is this different from merge sort, where items only reach their final positions at the very end?

*Your answer:*

---
## Experiment 2: Breaking the Barrier

This is the big moment. We'll run the simple O(n^2) sorts from Lab 11 alongside your Lab 12 sorts and watch the curves diverge.

The bubble sort and insertion sort counted versions are provided below so the notebook is self-contained — you don't need your Lab 11 code.

In [ ]:
# ── Lab 11 sorts (provided for comparison) ───────────────────────

def bubble_sort_counted(a_list):
    """Bubble sort with comparison and exchange counting."""
    comparisons = 0
    exchanges = 0
    for i in range(len(a_list) - 1):
        for j in range(len(a_list) - 1 - i):
            comparisons += 1
            if a_list[j] > a_list[j + 1]:
                a_list[j], a_list[j + 1] = a_list[j + 1], a_list[j]
                exchanges += 1
    return (a_list, comparisons, exchanges)


def insertion_sort_counted(a_list):
    """Insertion sort with comparison and data-move counting."""
    comparisons = 0
    data_moves = 0
    for i in range(1, len(a_list)):
        current_value = a_list[i]
        position = i - 1
        while position >= 0 and a_list[position] > current_value:
            comparisons += 1
            a_list[position + 1] = a_list[position]
            data_moves += 1
            position -= 1
        if position >= 0:
            comparisons += 1  # the comparison that ended the while
        a_list[position + 1] = current_value
        data_moves += 1
    return (a_list, comparisons, data_moves)

In [ ]:
import random
import matplotlib.pyplot as plt

sizes = [100, 500, 1000, 5000, 10000]

# Storage for results
results = {name: {"comps": [], "moves": []} for name in
           ["Bubble", "Insertion", "Merge", "Quick"]}

for n in sizes:
    data = list(range(n))
    random.shuffle(data)

    _, bc, be = bubble_sort_counted(data[:])
    _, ic, im = insertion_sort_counted(data[:])
    _, mc, mm = merge_sort_counted(data[:])
    _, qc, qe = quick_sort_counted(data[:])

    results["Bubble"]["comps"].append(bc)
    results["Bubble"]["moves"].append(be)
    results["Insertion"]["comps"].append(ic)
    results["Insertion"]["moves"].append(im)
    results["Merge"]["comps"].append(mc)
    results["Merge"]["moves"].append(mm)
    results["Quick"]["comps"].append(qc)
    results["Quick"]["moves"].append(qe)

    print(f"n={n:>6}: Bubble={bc:>12,}  Insertion={ic:>12,}  Merge={mc:>12,}  Quick={qc:>12,}")

In [ ]:
colors = {"Bubble": "#e74c3c", "Insertion": "#e67e22",
          "Merge": "#2ecc71", "Quick": "#3498db"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Comparisons
for name in results:
    ax1.plot(sizes, results[name]["comps"], 'o-',
             label=name, color=colors[name])
ax1.set_xlabel('List Size (n)')
ax1.set_ylabel('Comparisons')
ax1.set_title('Comparisons: O(n^2) vs O(n log n)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Data moves
for name in results:
    ax2.plot(sizes, results[name]["moves"], 'o-',
             label=name, color=colors[name])
ax2.set_xlabel('List Size (n)')
ax2.set_ylabel('Data Moves')
ax2.set_title('Data Movement: O(n^2) vs O(n log n)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Experiment 2 Questions

**Q1:** At what list size do the O(n^2) and O(n log n) curves visibly diverge? Is the separation gradual or does it suddenly explode?

*Your answer:*


**Q2:** At n = 10,000, roughly how many times more comparisons does bubble sort make than merge sort? Than quicksort?

*Your answer:*


**Q3:** Shell sort isn't shown on this graph (it doesn't have a counted version). Based on what you know about its complexity — between O(n) and O(n^2) — where would you expect it to fall? Closer to the red/orange lines or the green/blue lines?

*Your answer:*

---
## Experiment 3: Quicksort's Achilles Heel

Merge sort is O(n log n) in **all** cases. Quicksort is O(n log n) on **average** but O(n^2) in the **worst** case. Let's find that worst case.

In [ ]:
n = 1000

already_sorted = list(range(n))
reverse_sorted = list(range(n - 1, -1, -1))
random_list = list(range(n))
random.shuffle(random_list)

cases = {
    "Already sorted": already_sorted,
    "Reverse sorted": reverse_sorted,
    "Random": random_list,
}

print(f"{'Case':<20} {'Algorithm':<12} {'Comparisons':>12} {'Data Moves':>12}")
print("-" * 60)

for case_name, data in cases.items():
    _, mc, mm = merge_sort_counted(data[:])
    _, qc, qe = quick_sort_counted(data[:])
    print(f"{case_name:<20} {'Merge':<12} {mc:>12,} {mm:>12,}")
    print(f"{'':<20} {'Quick':<12} {qc:>12,} {qe:>12,}")
    if case_name != "Random":
        print()
    else:
        print()

### Experiment 3 Questions

**Q1:** On random input, which makes fewer comparisons — merge sort or quicksort? Are they in the same ballpark?

*Your answer:*


**Q2:** On already-sorted input, what happens to quicksort's comparison count compared to random input? How does it compare to merge sort on the same input?

*Your answer:*


**Q3:** Why does sorted input cause quicksort's worst case? Think about what happens when the pivot (first item) is always the smallest value — what do the partitions look like? How many levels of recursion do you get?

*Your answer:*


**Q4:** Given this weakness, why would anyone use quicksort instead of merge sort? Think about the data moves column — what does "in-place" mean for memory when sorting a list of 10 million items?

*Your answer:*